In [ ]:
# Select an existing Processing run. Read its actual inputs, including cache references.
# Print full S3 assignments that can be copied into a Python script.
# This inspects AWS metadata only; it does not run or change the pipeline.
MODEL_ID = "usecase2"
PIPELINE_KIND = "testing"  # "testing" or "inference"
EXECUTION_ID = "qaq3ddxjfasy"  # Historical testing example, November 2, 2026.
# Matching historical inference example: PIPELINE_KIND = "inference"; EXECUTION_ID = "brf4bz1lunno"
# Those examples belong to namespace src1007cced-i1102a; use a run from your configured namespace.

from pathlib import Path
import boto3
import json

REPOSITORY = next(p for p in (Path.cwd(), *Path.cwd().parents)
                  if (p / "mlops-models/src/mlops_models").is_dir())
config = json.loads((REPOSITORY / "notebooks/config.json").read_text())
if PIPELINE_KIND not in {"testing", "inference"}:
    raise ValueError("Choose testing or inference.")
sm = boto3.client("sagemaker")
pipeline = sm.describe_pipeline(PipelineName=f"{config['namespace']}-{PIPELINE_KIND}")
execution_arn = pipeline["PipelineArn"] + "/execution/" + EXECUTION_ID

# Read Postprocess's job rather than guessing where cached producer outputs live.
lookup_arn = execution_arn
visited = set()
while True:
    if lookup_arn in visited:
        raise ValueError("Repeated cache reference: " + lookup_arn)
    visited.add(lookup_arn)
    steps = [step for page in sm.get_paginator("list_pipeline_execution_steps").paginate(
        PipelineExecutionArn=lookup_arn) for step in page["PipelineExecutionSteps"]]
    step = next((step for step in steps if step["StepName"] == "Postprocess"), None)
    if step is None:
        raise ValueError("No Postprocess step found for " + lookup_arn)
    job_arn = step.get("Metadata", {}).get("ProcessingJob", {}).get("Arn")
    if job_arn:
        break
    cached_from = step.get("CacheHitResult", {}).get("SourcePipelineExecutionArn")
    if not cached_from:
        raise ValueError("Postprocess has no Processing job yet. Run it once before inspecting its inputs.")
    lookup_arn = cached_from

job = sm.describe_processing_job(ProcessingJobName=job_arn.rsplit("/", 1)[1])
environment = job.get("Environment", {})
if environment.get("MODEL_ID") != MODEL_ID:
    raise ValueError(f"This job belongs to {environment.get('MODEL_ID')!r}, not {MODEL_ID!r}.")
if environment.get("MLOPS_PREDICTION_FORMAT", "csv") != "csv":
    raise ValueError("This notebook uses Processing/CSV scoring, not native Batch Transform.")
mounts = {item["S3Input"]["LocalPath"].rstrip("/"): item["S3Input"]["S3Uri"].rstrip("/")
          for item in job["ProcessingInputs"] if "S3Input" in item}
DATA_URI = mounts["/opt/ml/processing/raw"] + "/data.csv"
SELECTION_URI = mounts["/opt/ml/processing/selection"] + "/selected_features.csv"
MODEL_URI = mounts["/opt/ml/processing/model"]
SCORE_OUTPUT_URI = mounts["/opt/ml/processing/input"] + "/"
REGION = sm.meta.region_name

# Copy this output, then cell 2, under the entrypoint's bottom main guard.
# A training ID is unnecessary: MODEL_URI is the checkpoint actually passed to this job.
print(f"# {PIPELINE_KIND.upper()} | model={MODEL_ID} | execution={EXECUTION_ID}")
print(f"# Job: {job_arn}")
print("# DATA_URI = original GetData rows; SELECTION_URI = Preprocess's copy from the model.")
print("# SCORE_OUTPUT_URI = all Score files; MODEL_URI = the resolved training model archive.")
for name in ("MODEL_ID", "PIPELINE_KIND", "REGION", "DATA_URI", "SELECTION_URI",
             "MODEL_URI", "SCORE_OUTPUT_URI"):
    print(f"{name} = {globals()[name]!r}")
print(f"REPOSITORY = {str(REPOSITORY)!r}")


In [ ]:
# Download the raw rows, selection and model; copy every Score file with its relative path.
# Unpack the model and set the environment used by model code.
# Call the local checkout's entrypoint; keep results in a new local output directory.
from pathlib import Path
from tempfile import mkdtemp
from urllib.parse import urlsplit
import importlib
import json
import os
import sys
import tarfile
import boto3

REPOSITORY = Path(REPOSITORY)
sys.path.insert(0, str(REPOSITORY / "mlops-models/src"))
work_directory = Path(mkdtemp(prefix="postprocess-"))
data_directory = work_directory / "raw"
selection_file = work_directory / "selection/selected_features.csv"
model_archive = work_directory / "model.tar.gz"
model_directory = work_directory / "model"
prediction_directory = work_directory / "input"
output_directory = work_directory / "output"
output_directory.mkdir()

# Local model reads and writable output; the entrypoint does not upload run reports.
os.environ.update({
    "MODEL_ID": MODEL_ID,
    "MLOPS_MODEL_DIR": str(model_directory),
    "SM_MODEL_DIR": str(model_directory),
    "SM_OUTPUT_DATA_DIR": str(output_directory),
    "SM_OUTPUT_DIR": str(output_directory),
    "MLOPS_PREDICTION_FORMAT": "csv",
    "MLOPS_MODEL_ARTIFACT_URI": MODEL_URI,
    "MLOPS_WRITE_RUN_REPORT": "0",
})

# Download exact files. DATA_URI is raw, not Preprocess/prepared/data.csv.
s3 = boto3.client("s3", region_name=REGION)
for uri, destination in (
    (DATA_URI, data_directory / "data.csv"),
    (SELECTION_URI, selection_file),
    (MODEL_URI, model_archive),
):
    print(f"Download: {uri}", flush=True)
    destination.parent.mkdir(parents=True, exist_ok=True)
    address = urlsplit(uri)
    s3.download_file(address.netloc, address.path.lstrip("/"), str(destination))

# Copy the whole Score folder, including messaging, importance files and nested output/.
address = urlsplit(SCORE_OUTPUT_URI)
prefix = address.path.lstrip("/").rstrip("/") + "/"
score_files = 0
for page in s3.get_paginator("list_objects_v2").paginate(Bucket=address.netloc, Prefix=prefix):
    for item in page.get("Contents", []):
        key = item["Key"]
        if key.endswith("/"):
            continue
        destination = (prediction_directory / key[len(prefix):]).resolve()
        destination.relative_to(prediction_directory.resolve())
        destination.parent.mkdir(parents=True, exist_ok=True)
        s3.download_file(address.netloc, key, str(destination))
        score_files += 1
print(f"Score files: {score_files} | {prediction_directory}", flush=True)

# The entrypoint receives an extracted directory, not model.tar.gz.
with tarfile.open(model_archive) as archive:
    archive.extractall(model_directory, filter="data")
print(json.dumps({
    "model_id": MODEL_ID,
    "data_directory": str(data_directory),
    "selection_file": str(selection_file),
    "input_path": str(prediction_directory),
    "model_dir": str(model_directory),
    "output_dir": str(output_directory),
}, indent=2), flush=True)

# Execute. When pasted inside the entrypoint itself, remove the import/reload block
# below and change postprocess.execute(...) to execute(...).
if PIPELINE_KIND == "testing":
    from mlops_models.entrypoints import postprocess_test as postprocess
else:
    from mlops_models.entrypoints import postprocess_inference as postprocess
postprocess = importlib.reload(postprocess)
result_directory = postprocess.execute(
    model_id=MODEL_ID,
    input_path=prediction_directory,
    output_dir=output_directory,
    model_dir=model_directory,
    data_directory=data_directory,
    selection_file=selection_file,
)
print(result_directory)
